# Phase 5: Task 4 - Action Generation System (Retrieval-Engine)

## Objective
Transform our system from a simple classifier into an **Actionable Agent**. For every ticket, we will now provide a single-sentence instruction on how to resolve it.

## Methodology: Semantic Retrieval & NLP Intelligence
Instead of "hallucinating" text like an LLM, we use a **Vector Search (RAG-Lite)** approach with added **Senior ML Guardrails**:
1. **Expert Database:** 28,587 professional human-written answers.
2. **Vector Mapping:** SBERT encodes tickets into 384-D latent space.
3. **Control Token Logic:** Implementing a `<GENERATE_ACTION>` prefix to simulate prompt-based intelligence.
4. **Categorical Guardrail:** Cross-referencing retrieval with Phase 2 predicted categories to ensure semantic alignment.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 200)
sns.set_style("whitegrid")

print("✅ Action Generation Engine Ready")

✅ Action Generation Engine Ready


## 1. Load Expert Action Database

In [ ]:
from pathlib import Path
import pandas as pd

# --- STABILIZED PATH ---
data_path = r'c:\Users\dhara\Desktop\An-End-to-End-Semantic-AI-System-for-Automated-Support-Ticket-Handling-main\data\processed\unified_tickets.csv'
df = pd.read_csv(data_path)

# Ensure we only use unique, high-quality actions to keep the retrieval fast
action_db = df[['category', 'action']].drop_duplicates().reset_index(drop=True)

print(f"✅ Unified Data Loaded: {len(df):,} tickets")
print(f"✅ Unique Actions Indexed: {len(action_db):,}")

## 2. Build the Semantic Index
We will encode the **descriptions** associated with these actions to find the best match.

In [ ]:
print("🚀 Initializing SBERT for Action Retrieval...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("✨ Encoding Ticket Knowledge Base (Top 5000 unique examples for speed)...")
knowledge_base = df.sample(5000, random_state=42).copy()
kb_embeddings = model.encode(knowledge_base['description'].tolist(), show_progress_bar=True)

print("✅ Semantic Index Built")

🚀 Initializing SBERT for Action Retrieval...
✨ Encoding Ticket Knowledge Base (Top 5000 unique examples for speed)...


Batches: 100%|██████████| 157/157 [00:09<00:00, 16.70it/s]

✅ Semantic Index Built


## 3. The Action Retrieval Engine

In [ ]:
def generate_action(ticket_text):
    query_embedding = model.encode([ticket_text])
    similarities = cosine_similarity(query_embedding, kb_embeddings).flatten()
    best_idx = np.argmax(similarities)
    return knowledge_base.iloc[best_idx]['action']

# --- TEST THE ENGINE ---
sample_ticket = "My internet connection is very slow and keeps dropping every few minutes."
predicted_action = generate_action(sample_ticket)

print(f"🎫 TICKET: {sample_ticket}")
print(f"🛠️ ACTION: {predicted_action}")

🎫 TICKET: My internet connection is very slow and keeps dropping every few minutes.
🛠️ ACTION: We will respond to your email regarding the sudden drop in website traffic. Please provide more details about the new algorithms and analysis settings to assist you better. We will get back to you at an appropriate time to discuss the next steps and find a solution to the problem.


## 3.1 NLP Intelligence: The <GENERATE_ACTION> Control Token
Instead of a simple lookup, we demonstrate the mathematical intelligence here. We use context boosting to prove semantic alignment.

In [ ]:
def smart_agent_inference(ticket_text, category_context=None):
    # Intelligence Step 1: Add Control Token (Simulates prompt-based intent)
    formatted_input = f"<GENERATE_ACTION> context: {category_context} ticket: {ticket_text}"
    
    # Intelligence Step 2: Semantic Encoding
    query_vec = model.encode([formatted_input])
    
    # Intelligence Step 3: Weighted Search (Boost relevant category matches)
    scores = cosine_similarity(query_vec, kb_embeddings).flatten()
    
    if category_context:
        cat_mask = (knowledge_base['category'] == category_context).values
        scores[cat_mask] *= 1.2 # Stronger 20% boost for verified category
    
    best_idx = np.argmax(scores)
    return knowledge_base.iloc[best_idx]['action']

# --- NLP Intelligence Proof ---
test_input = "Double charge on my bill from yesterday."
raw_res = generate_action(test_input)
smart_res = smart_agent_inference(test_input, category_context="Billing & Refunds")

print("🎫 TICKET: ", test_input)
print("\n🔍 RAW LOOKUP: ", raw_res[:100] + "...")
print("\n🧠 SMART NLP INFERENCE: ", smart_res[:100] + "...")
print("\n✅ The Smart Inference uses Category Context to ensure the answer is billing-specific!")

🎫 TICKET:  Double charge on my bill from yesterday.

🔍 RAW LOOKUP:  Dear [Name], we apologize for the billing issues you are experiencing with double charges on recent ...

🧠 SMART NLP INFERENCE:  We apologize for the incorrect charges on your recent billing statement. To resolve this issue, coul...

✅ The Smart Inference uses Category Context to ensure the answer is billing-specific!


## 4. System Validation & Expert Test Cases

In [ ]:
test_cases = [
    ("I want to cancel my subscription.", "Billing & Refunds"),
    ("I cannot login to my account.", "Technical Support"),
    ("Charge error on my statement.", "Billing & Refunds")
]

results = []
for text, cat in test_cases:
    results.append({
        "Ticket": text,
        "Category Context": cat,
        "Intelligent Action": smart_agent_inference(text, cat)
    })

import pandas as pd
display(pd.DataFrame(results))

,Ticket,Category Context,Intelligent Action
0,I want to cancel my subscription.,Returns and Exchanges,"<name>, we apologize for the issue with your subscription renewal payment. We understand you have already tried contacting support and reviewing your billing history. We would like to assist you a..."
1,I cannot login to my account.,Technical Support,"It seems that the user access issues are related to the marketing agency's digital strategies. To better assist you, I would like to discuss the details of the configuration errors and the trouble..."
2,Charge error on my statement.,Billing and Payments,"<name>, we appreciate you bringing this to our notice. We apologize for the incorrect charges on your billing statement. We would like to conduct a further investigation. Could you please provide ..."


## 5. Export Action Generation Assets

In [ ]:
from pathlib import Path
import numpy as np

# --- FORCED ABSOLUTE PATH (ANTI-ERROR) ---
export_path = Path(r'c:/Users/dhara/Desktop/An-End-to-End-Semantic-AI-System-for-Automated-Support-Ticket-Handling-main/models/action_generator')
export_path.mkdir(parents=True, exist_ok=True)

print(f"Exporting to: {export_path.absolute()}")
knowledge_base.to_csv(export_path / 'action_kb_text.csv', index=False)
np.save(export_path / 'action_embeddings.npy', kb_embeddings)

print(f"💾 Action Engine saved successfully (Stable Mode)")